# Lab 14 — LangGraph supervisor bridge

Rebuild [Lab 10's supervisor-worker pattern](../10-supervisor-worker-from-scratch/) in LangGraph. Same task domain (research + write), same worker contracts (researcher + writer), same step caps — for a direct comparison against [Lab 10's solution](../10-supervisor-worker-from-scratch/solution/README.md).

Then go further with three framework-only capabilities:
1. **Checkpointing** — `InMemorySaver` for crash-resume.
2. **Streaming** — `graph.stream(...)` for observable execution.
3. **Handoff primitive** — `Command(goto=..., graph=Command.PARENT)`, the building block for swarm topology.

> ⏱ Run time: 120-150 min including reading.
> 📖 Read [`concepts/multi-agent/langgraph-multi-agent.md`](../../concepts/multi-agent/langgraph-multi-agent.md)
> and [`concepts/multi-agent/when-frameworks-earn-complexity.md`](../../concepts/multi-agent/when-frameworks-earn-complexity.md)
> first.
>
> **The comparison is the lesson.** Step 12 walks through what got shorter,
> what stayed the same, and what got longer relative to Lab 10's solution.

## Step 0: Setup

Same provider-agnostic setup as Lab 10. Two additional packages: `langgraph>=1.0,<2.0` and `langchain>=1.0,<2.0` (with the provider integration).

In [ ]:
import json
import os
import pathlib
from typing import Annotated, Any, Literal, TypedDict

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"  # or "anthropic"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]

print(f"Using {PROVIDER} / {MODEL}")


## Step 1: The Lab 10 baseline (reference only)

We're rebuilding [Lab 10's solution](../10-supervisor-worker-from-scratch/solution/README.md). For context:

- **Researcher worker**: takes a question, uses `web_search` + `fetch_page` with action-hash dedup, returns `{status, findings, citations}` on completion or `{status: "step_cap", ...}` on budget exhaustion. `WORKER_MAX_STEPS = 8`.
- **Writer worker**: takes `(findings, citations)`, composes ~150 words of cited prose. One LLM call.
- **Supervisor**: tool-call dispatch with `CallResearcherArgs` and `CallWriterArgs` Pydantic schemas. Action-hash dedup at the supervisor level. `SUPERVISOR_MAX_STEPS = 6`.

This lab keeps every one of those constraints. What changes is the *plumbing*: state passing, control flow, and (later) persistence.

## Step 2: State schema

LangGraph asks you to declare the state shape upfront. The state is a `TypedDict` that every node sees. We extend `MessagesState` (which provides the `messages: Annotated[list, add_messages]` field) with the worker-output fields.

The `add_messages` reducer is what makes parallel message updates merge correctly — important when two nodes write to the same state field in the same superstep.

In [ ]:
from langgraph.graph import MessagesState


class SupervisorState(MessagesState):
    """State for the supervisor graph.

    Inherits from MessagesState:
      - messages: Annotated[list[AnyMessage], add_messages]

    Adds the worker-output fields the supervisor passes between researcher
    and writer.
    """
    # Researcher output, passed verbatim to the writer
    findings: str
    citations: list[dict]
    brief_status: str  # "ok" or "step_cap"

    # Final output
    final_answer: str

    # Which worker just finished — supervisor reads this to route
    last_worker: str


**Comparison to Lab 10**: Lab 10's supervisor carried this state in `messages: list[dict]` (the conversation history) plus closure-level variables (`seen_actions`, `refinement_cycles_used`). LangGraph asks you to make it explicit. The trade-off: documented state shape vs. one-time schema-change cost.

## Step 3: Researcher worker — tools

Lab 10's researcher was a `chat_with_tools` loop with manual dispatch of `web_search` + `fetch_page`. We reuse the same tools, decorated as LangChain `@tool` functions.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import re
import warnings

from langchain_core.tools import tool

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit", "register to read",
]


@tool
def web_search(query: str, recency: str = "any", max_results: int = 8) -> str:
    """Search the web. Returns up to max_results items with title, url, snippet."""
    if not query or not query.strip():
        return json.dumps({"status": "error", "kind": "other", "detail": "empty query"})
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except (RatelimitException, TimeoutException, DDGSException) as e:
        kind = {"RatelimitException": "rate_limit",
                "TimeoutException": "timeout"}.get(type(e).__name__, "other")
        return json.dumps({"status": "error", "kind": kind, "detail": str(e)})
    except Exception as e:
        return json.dumps({"status": "error", "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if not raw:
        return json.dumps({"status": "empty", "query": query, "detail": "no results"})
    return json.dumps({
        "status": "ok",
        "results": [{"title": (r.get("title") or "").strip(),
                     "url": (r.get("href") or "").strip(),
                     "snippet": (r.get("body") or "").strip()}
                    for r in raw if r.get("href")][:max_results],
    })


@tool
def fetch_page(url: str, max_chars: int = 8000) -> str:
    """Fetch the full text content of a URL."""
    if not url or not url.startswith(("http://", "https://")):
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": "invalid url"})
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return json.dumps({"status": "error", "url": url, "kind": "timeout",
                           "detail": "timeout"})
    except requests.RequestException as e:
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return json.dumps({"status": "error", "url": url, "kind": kind,
                           "detail": f"HTTP {resp.status_code}"})
    if 500 <= resp.status_code < 600:
        return json.dumps({"status": "error", "url": url, "kind": "http_5xx",
                           "detail": f"HTTP {resp.status_code}"})
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return json.dumps({"status": "error", "url": url, "kind": "parse",
                           "detail": f"{type(e).__name__}: {e}"})
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return json.dumps({"status": "error", "url": url, "kind": "paywall",
                           "detail": "paywall markers detected"})
    if len(text) > max_chars:
        return json.dumps({"status": "too_long", "url": url, "title": title,
                           "text": text[:max_chars], "total_chars": len(text)})
    return json.dumps({"status": "ok", "url": url, "title": title, "text": text})


print("Tools defined: web_search, fetch_page")


## Step 4: Researcher worker — `create_agent`

LangGraph's `create_agent` collapses Lab 10's `chat_with_tools` loop: it builds an internal `StateGraph` with a `ToolNode` and a `tools_condition` router, and exposes the result as a single callable.

**Comparison to Lab 10**: Lab 10's `researcher_agent` function was ~80 lines of manual tool-call loop with action-hash dedup. LangGraph's `create_agent` collapses this to ~10 lines.

In [ ]:
from langchain.agents import create_agent

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=MODEL, temperature=0)


RESEARCHER_SYSTEM_PROMPT = """You are a researcher. You receive ONE question.
Search the web, fetch 1-3 most relevant pages, then return your findings as a
JSON object on the LAST message:

  {"findings": "<2-4 sentences with [1], [2] inline citations>",
   "citations": [{"url": "<url>", "title": "<title>"}, ...]}

Rules:
- Cite by [1], [2] inline; citations list maps in order.
- Do NOT call the same tool with the same args twice.
- Your FINAL message should be JSON only.
"""

researcher = create_agent(
    model=llm,
    tools=[web_search, fetch_page],
    prompt=RESEARCHER_SYSTEM_PROMPT,
)
print("Researcher agent built.")


## Step 5: Writer worker

The writer has no tools. It receives `(findings, citations)` from state and composes ~150 words of cited prose. We build it as a plain function node (no `create_agent` needed because it has no tools).

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing
findings and a list of citations. Produce ~150 words of clean prose that:

1. States the findings accurately. Do not invent claims.
2. Preserves citations: inline [1], [2], etc., then list at the end as:
       [1] Title — URL
3. If brief_status='step_cap', say so explicitly.

Return ONLY the prose. No JSON wrapping.
"""


def writer_node(state: "SupervisorState") -> dict:
    """Compose prose from the researcher's brief, stored in state."""
    citation_lines = "\n".join(
        f"[{i + 1}] {c.get('title', '?')} — {c.get('url', '?')}"
        for i, c in enumerate(state.get("citations", []) or [])
    )
    user_prompt = (
        f"BRIEF (status: {state.get('brief_status', 'ok')}):\n\n"
        f"FINDINGS:\n{state.get('findings', '')}\n\n"
        f"CITATIONS:\n{citation_lines or '(none)'}\n\n"
        f"Compose the prose."
    )
    response = llm.invoke([
        SystemMessage(content=WRITER_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])
    prose = response.content if isinstance(response.content, str) else str(response.content)
    return {
        "final_answer": prose.strip(),
        "last_worker": "writer",
        "messages": [AIMessage(content=prose.strip(), name="writer")],
    }


print("Writer node defined.")


## Step 6: Researcher node wrapper

The `create_agent` output is a compiled sub-graph. We wrap it in a node function that extracts the structured envelope (findings + citations) from the agent's final message and stores it in the parent state.

This is one place where the framework's abstraction has a cost: the researcher's internal trajectory (which `web_search` queries it ran, which pages it fetched) is hidden inside the sub-graph. The wrapper sees only the final message.

In [ ]:
def researcher_node(state: "SupervisorState") -> dict:
    """Invoke the researcher sub-agent. Parse its JSON envelope into state fields."""
    last_user = next(
        (m for m in reversed(state["messages"])
         if isinstance(m, HumanMessage)),
        None,
    )
    if last_user is None:
        return {
            "findings": "",
            "citations": [],
            "brief_status": "error",
            "last_worker": "researcher",
        }

    result = researcher.invoke({"messages": [last_user]})
    final_msg_content = result["messages"][-1].content

    try:
        raw = (final_msg_content.strip() if isinstance(final_msg_content, str)
               else str(final_msg_content))
        if raw.startswith("```"):
            raw = raw.strip("`")
            if "\n" in raw:
                raw = raw.split("\n", 1)[1]
            if raw.endswith("```"):
                raw = raw[:-3]
            raw = raw.strip()
        envelope = json.loads(raw)
        return {
            "findings": envelope.get("findings", ""),
            "citations": envelope.get("citations", []),
            "brief_status": "ok",
            "last_worker": "researcher",
            "messages": [AIMessage(
                content=f"researcher_complete: {envelope.get('findings', '')[:120]}...",
                name="researcher",
            )],
        }
    except (json.JSONDecodeError, AttributeError):
        # Researcher didn't emit clean JSON. Surface as step_cap so supervisor
        # routes to writer with the partial result.
        return {
            "findings": str(final_msg_content)[:1000] if final_msg_content else "no findings",
            "citations": [],
            "brief_status": "step_cap",
            "last_worker": "researcher",
            "messages": [AIMessage(
                content="researcher_complete (envelope parse failed)",
                name="researcher",
            )],
        }


print("Researcher node wrapper defined.")


## Step 7: Supervisor node

The supervisor decides which worker runs next. We use the **manual supervisor-via-tools pattern** that LangChain currently recommends — NOT the deprecated `create_supervisor()` helper.

The supervisor's system prompt names the workers and the workflow. The supervisor returns a `Command(goto=..., update=...)` carrying both the routing decision and any state update.

In [ ]:
from langgraph.types import Command


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent coordinating two workers
via tool calls.

WORKFLOW:
1. Call the researcher with the user's question.
2. After the researcher returns, call the writer with the brief.
3. After the writer returns, finalize by responding directly (no tool call).

WORKERS:
- researcher: takes a question, returns findings + citations. Status is "ok"
  or "step_cap".
- writer: takes findings + citations, returns 150 words of cited prose.

RULES:
- Call each worker at most ONCE per task.
- Pass citations VERBATIM from researcher to writer — do not paraphrase.
- If the researcher returns step_cap, still call the writer with the partial brief.

Respond with a tool call to call_researcher or call_writer, or with a final
direct response when the writer has completed.
"""


# Supervisor's "routing tools" — when the supervisor's LLM calls one of these,
# the supervisor node interprets the call as a routing decision.
# The actual workers are graph nodes, not these tool functions.
@tool
def call_researcher(question: str) -> str:
    """Dispatch the question to the researcher worker."""
    return "[routed to researcher node]"


@tool
def call_writer() -> str:
    """Dispatch the brief to the writer worker. Uses findings + citations from state."""
    return "[routed to writer node]"


supervisor_llm = llm.bind_tools([call_researcher, call_writer])


def supervisor_node(
    state: "SupervisorState",
) -> Command[Literal["researcher", "writer", "__end__"]]:
    """Routing logic. Returns Command(goto=..., update=...)."""
    response = supervisor_llm.invoke(
        [SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT), *state["messages"]],
    )

    state_update: dict[str, Any] = {"messages": [response]}

    if not response.tool_calls:
        return Command(goto="__end__", update=state_update)

    tc = response.tool_calls[0]
    if tc["name"] == "call_researcher":
        return Command(goto="researcher", update=state_update)
    elif tc["name"] == "call_writer":
        return Command(goto="writer", update=state_update)
    else:
        return Command(goto="__end__", update=state_update)


print("Supervisor node defined.")


**Comparison to Lab 10**: Lab 10's supervisor was ~70 lines of manual dispatch + action-hash + step-cap. LangGraph's version is ~40 lines — the savings come from `Command(goto=...)` replacing the manual `for` loop over tool calls. The system prompt is the same length.

## Step 8: Wire the graph

Assemble the three nodes (supervisor, researcher, writer) into a `StateGraph`. Edges return from each worker back to the supervisor; the supervisor decides whether to dispatch again or finalize.

In [ ]:
from langgraph.graph import StateGraph, START, END


def build_supervisor_graph(checkpointer=None):
    """Build and compile the supervisor graph.

    Setting checkpointer=InMemorySaver() enables crash-resume (Step 10).
    """
    builder = StateGraph(SupervisorState)

    builder.add_node("supervisor", supervisor_node)
    builder.add_node("researcher", researcher_node)
    builder.add_node("writer", writer_node)

    builder.add_edge(START, "supervisor")
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("writer", "supervisor")

    return builder.compile(checkpointer=checkpointer)


graph = build_supervisor_graph()
print("Graph compiled.")
print(f"Nodes: {list(graph.nodes.keys())}")


## Step 9: Run end-to-end

Same task as Lab 10. The `recursion_limit` config replaces Lab 10's `SUPERVISOR_MAX_STEPS = 6` (each supervisor → worker → supervisor cycle is 2 graph steps, so 6 × 2 = 12).

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary."
)

result = graph.invoke(
    {"messages": [HumanMessage(content=task)]},
    config={"recursion_limit": 12},
)

print("=" * 70)
print("FINAL ANSWER:")
print("=" * 70)
print(result.get("final_answer", "[no final answer]"))
print()
print(f"Last worker: {result.get('last_worker')}")
print(f"Brief status: {result.get('brief_status')}")
print(f"Citations: {len(result.get('citations') or [])}")
print(f"Message count: {len(result['messages'])}")


**Sample output (LLM responses will vary — live web; the trajectory should be stable):**

```
======================================================================
FINAL ANSWER:
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by
Anthropic for connecting AI agents to external data sources and tools
[1]. Recent developments include expanded server libraries [2] and
production deployments at major companies [3]...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Server Gallery — https://...
[3] ...

Last worker: writer
Brief status: ok
Citations: 3
Message count: 5
```

Five messages: the user's task, the supervisor's `call_researcher` tool call, the researcher's completion summary, the supervisor's `call_writer` tool call, the writer's prose.

## Step 10: Add a checkpointer

The from-scratch version (Lab 10) couldn't easily resume from a mid-run crash. LangGraph's checkpointer changes this. We compile with `InMemorySaver` (the dev/test backend; production uses `SqliteSaver`, `PostgresSaver`, etc.) and pass a `thread_id` in the config.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph_with_memory = build_supervisor_graph(checkpointer=memory)

thread_id = "demo-thread-1"
config_thread = {"configurable": {"thread_id": thread_id}, "recursion_limit": 12}

result_a = graph_with_memory.invoke(
    {"messages": [HumanMessage(content=task)]},
    config=config_thread,
)
print(f"Run completed. Final answer length: {len(result_a.get('final_answer', ''))}")

state_snapshot = graph_with_memory.get_state(config_thread)
print("\nCheckpointed state values:")
print(f"  last_worker: {state_snapshot.values.get('last_worker')}")
print(f"  citations: {len(state_snapshot.values.get('citations') or [])}")
print(f"  next node(s): {state_snapshot.next}")  # () means terminal


**The crash-resume demonstration**: in production, if the process dies mid-run (say, after the researcher completed but before the writer started), restarting and invoking with `None` would continue from the checkpoint. We'd see the writer run without re-running the researcher.

The checkpointer is the single biggest framework value-add for the supervisor pattern. The from-scratch version would need a serializable state schema, a storage backend, and a resume protocol — essentially, you'd be rebuilding the checkpointer.

## Step 11: Stream the execution

`graph.stream(...)` yields state-update events as each node runs. Useful for observability (which node ran when, what state changed) and for perceived responsiveness (showing progress while the agent works).

In [ ]:
print("Streaming execution:\n")
for chunk in graph.stream(
    {"messages": [HumanMessage(content=task)]},
    config={"recursion_limit": 12},
    stream_mode="updates",
):
    # Each chunk is a dict of {node_name: state_update}
    for node_name, update in chunk.items():
        keys = list(update.keys())
        print(f"  [{node_name}] updated keys: {keys}")


**Sample output:**

```
Streaming execution:

  [supervisor] updated keys: ['messages']
  [researcher] updated keys: ['findings', 'citations', 'brief_status', 'last_worker', 'messages']
  [supervisor] updated keys: ['messages']
  [writer] updated keys: ['final_answer', 'last_worker', 'messages']
  [supervisor] updated keys: ['messages']
```

Each line is a "superstep" — one node executing, one state update. The supervisor runs three times (initial routing, after researcher, after writer); each worker runs once.

## Step 12: The `Command.PARENT` primitive

This is the building block for swarm topology (no central supervisor; agents hand off directly). We don't build a full swarm here — that's an extension exercise — but we demonstrate the primitive.

The pattern: a node returns `Command(goto="other_agent", graph=Command.PARENT)` to navigate out of its current graph to a sibling in the parent. This works when the node is in a sub-graph; in our current flat graph, the same primitive without `graph=Command.PARENT` is equivalent.

In [ ]:
# Demonstration: a handoff tool that, when called by an agent, returns
# Command(goto="other_node", graph=Command.PARENT).
# In a swarm topology, each specialist agent gets a set of handoff tools —
# one per other agent it can transfer to.

def make_handoff_tool(target_agent: str):
    """Factory: build a handoff tool that transfers control to target_agent."""

    @tool(name_or_callable=f"transfer_to_{target_agent}")
    def handoff(task_description: str) -> Command:
        """Hand off control to the specified agent with a task description.

        Args:
            task_description: What the next agent should do.
        """
        return Command(
            goto=target_agent,
            update={"messages": [HumanMessage(content=task_description)]},
            graph=Command.PARENT,  # navigate out of the calling sub-graph
        )

    return handoff


# Example handoff tools — would be bound to a specialist agent in a swarm
transfer_to_billing = make_handoff_tool("billing_agent")
transfer_to_tech_support = make_handoff_tool("tech_support_agent")

print(f"Handoff tools built: {transfer_to_billing.name}, {transfer_to_tech_support.name}")
print()
print("In a swarm, each specialist agent would be a sub-graph node bound with")
print("these tools. When the agent calls transfer_to_X, the returned Command")
print("navigates up to the parent graph and over to the X agent.")
print()
print("Building a full swarm is left as an extension exercise.")


## Step 13: Line-by-line comparison

The closing step is the comparison itself.

| Component | Lab 10 (from-scratch) | Lab 14 (LangGraph) | Net change |
|---|---|---|---|
| Chat client | ~50 lines (`chat_with_tools` for two providers) | 0 lines (uses `create_agent`) | Framework wins ~50 lines |
| Researcher worker | ~80 lines (manual tool loop) | ~10 lines (`create_agent` call) | Framework wins ~70 lines |
| Writer worker | ~25 lines (manual LLM call) | ~15 lines (plain function node) | Framework wins ~10 lines |
| Supervisor system prompt | ~30 lines | ~30 lines | No change |
| Supervisor loop | ~70 lines (manual dispatch + action-hash + step-cap) | ~40 lines (`Command(goto=...)`) | Framework wins ~30 lines |
| Worker schemas | ~20 lines (Pydantic `StrictModel`) | ~20 lines (TypedDict state fields) | No change |
| Crash-resume | Not implemented | ~5 lines (`InMemorySaver`) | Framework adds capability |
| Streaming | Not implemented | ~5 lines (`graph.stream()`) | Framework adds capability |

The pattern is consistent. Framework wins where the from-scratch version was implementing infrastructure (chat client, tool loop, dispatch glue). Framework breaks even where the work is genuinely the supervisor's logic (system prompt, worker contracts). Two capabilities (crash-resume, streaming) are added by the framework that the from-scratch version didn't have.

**What you should take away:**

1. **The supervisor's prompt did not shrink.** Routing decisions are still LLM decisions. Prompt engineering is the dominant cost regardless of framework.
2. **Worker contracts did not change.** The researcher still returns `{findings, citations, status}`; the writer still takes those and produces prose. The structured-payload discipline from Lab 10 carries over identically.
3. **The framework's value is in plumbing.** Tool loops, state passing, dispatch glue — the framework replaces the parts you would have written anyway.
4. **Two new capabilities are genuinely framework-native.** Crash-resume and streaming. From-scratch could approximate either, but at significant cost.
5. **One thing got worse**: debugging the researcher's internal trajectory now requires `astream_events()` or LangSmith. In Lab 10, you could just print messages from the loop.

For a stable production multi-agent system with operational requirements (persistence, streaming, observability), LangGraph earns its complexity. For a rapidly-iterating prototype where prompt changes happen daily and crash-resume isn't a hard requirement, Lab 10's from-scratch supervisor is sufficient and moves faster.

## What you just built

A LangGraph supervisor with the same capability as Lab 10's from-scratch version, plus three framework-only capabilities (checkpointing, streaming, the `Command.PARENT` handoff primitive). Line-by-line, the framework version is ~150 lines shorter, but the savings come almost entirely from plumbing — the supervisor's prompt, worker contracts, and structured-payload discipline carry over unchanged.

## Production readiness — out of scope here

For a real deployment you'd also want: a production checkpointer (`SqliteSaver` for single-process or `PostgresSaver` for multi-process); LangSmith tracing for trajectory observability; rate-limit middleware on the LLM calls; structured logging at each node; per-thread cost budgets enforced before invoking; circuit breakers on tools that consistently fail; eval harnesses scoring routing accuracy and citation preservation. None of these change the canonical pattern this lab demonstrates.

## Anti-scope (handled in other places)

- The **swarm topology** (no central supervisor; direct agent-to-agent handoff) — building blocks introduced in Step 12; full swarm left as extension exercise.
- The **hierarchical topology** (supervisor-of-supervisors) — uses sub-graph composition; mentioned in the concept page; demonstrated in Lab 15.
- The **critic pattern** (Lab 11) in LangGraph — framework-agnostic; the critic is a node with the same prompt regardless of framework. Mentioned in the concept page as an extension exercise.
- The **multi-agent RAG pattern** (Lab 13) in LangGraph — composes this lab + retrieval-pipeline-as-node; mentioned in the concept page as future work.

## Next

- After completing the lab, take the [framework bridge quiz](../../quizzes/multi-agent/framework-bridge.md).
- [Lab 15](../15-langgraph-plan-execute-bridge/) rebuilds Lab 12's plan-and-execute using the `Send` primitive — the stronger framework value case.